# Annotate samples 

Use `pyannotations` for best ease to annotate samples. Note, the samples here are larger than the intended ones and additionally marked, as easier to annotate in context + may want to make a judgement call whether to treat as labelled or keep for the unsupervised portion.

In [1]:
# re-loads code before cell execution, so if sth changes it will propagate:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys, shutil
import logging

import time as tm
import numpy as np
import pandas as pd

from PIL import Image
from ipyannotations import images

sys.path.append('..')

import src.utils as ssut
import src.data_utils as sdut
import src.geometry as sgut

## Set up annnotation

In [3]:
data_path = '../data/xView/to_annotate/'
df = pd.read_csv(f'{data_path}labels.csv')

out_path_labels = '../data/xView/labels_1/'
os.makedirs(out_path_labels, exist_ok= True)

to_annot_names = [ x for x in os.listdir( data_path) if not x.endswith('.csv')]
to_annot = [os.path.join( data_path, name) for name in to_annot_names]

In [4]:
def on_finish( df, label_di, labels):
    assert set(df['image'].values) == set(label_di.keys())
    # this assertion will fail if more clicks than there is images, and thus break the loop:
    assert len(df) == len(labels)
    # map labels to image names:
    df['labels'] = df['image'].map( label_di)
    df.loc[ df['labels'] == 'road', 'roads'] = 1
    df.loc[ df['labels'] == 'no road', 'roads'] = 0
    # save:
    name_split = df['image'].str.split('_').map(lambda x: x[0]).unique()
    assert len(name_split)==1
    which_tiff = name_split[0]
    out_to_ = f'{out_path_labels}{which_tiff}.csv'
    print('------------------------------------------------------------------------------------------------------------')
    print(f'---------------------------  Outputting to: {out_to_}. --------------------')
    print('------------------------------------------------------------------------------------------------------------')
    df.to_csv( out_to_, index= False)

In [5]:
widget = images.ClassLabeller(
    options=['road','no road','unlabel','reject']
)
labels = []
label_di = {}

def annotate( annot):
    labels.append( annot)
    # to ensure correct matching of labels to file names:
    global next_ 
    label_di[ next_.split('/')[-1]] = annot
    try:
        next_ = to_annot.pop(0)
        widget.display( next_)
    except IndexError:
        print('------------------------------------------------------------------------------------------------------------')
        print('---------------------------- Aaaaand your task is done, ran out of images in the folder! -------------------')
        print('------------------------------------------------------------------------------------------------------------')
        on_finish( df, label_di, labels)
        
widget.on_submit( annotate)

## Annotate interactively


In [6]:
global next_ 
next_ = to_annot.pop(0)
widget.display( next_)
widget

ClassLabeller(children=(Box(children=(Output(layout=Layout(margin='auto', min_height='50px')),), layout=Layout…

------------------------------------------------------------------------------------------------------------
---------------------------- Aaaaand your task is done, ran out of images in the folder! -------------------
------------------------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------------------------
---------------------------  Outputting to: ../data/xView/labels_1/tif43.csv. --------------------
------------------------------------------------------------------------------------------------------------


### can sanity-check if wanting to...

In [29]:
label_di == dict(zip(to_annot_names,labels))

False

In [33]:
len(labels), len(to_annot_names)

(414, 414)

In [34]:
df.head(3)

,xs,ys,buildings,roads,cars,bbox_np,image,bbox_context,bbox_in_context,built_area,n_cars,n_bus_trucks,labels
0,70,258,-1,0,-1,"[6, 194, 134, 322]",tif43_x70_y258.png,"[0, 130, 256, 386]","[6, 64, 134, 192]",0.533020,1.0,0.0,no road
1,74,608,-1,1,-1,"[10, 544, 138, 672]",tif43_x74_y608.png,"[0, 480, 256, 736]","[10, 64, 138, 192]",0.564697,4.0,0.0,road
2,75,976,-1,-1,-1,"[11, 912, 139, 1040]",tif43_x75_y976.png,"[0, 848, 256, 1104]","[11, 64, 139, 192]",0.569641,8.0,1.0,unlabel


In [35]:
len(df.loc[ df['labels'] != 'reject']), len(df.loc[ df['labels'].str.contains('road')])

(261, 164)